In [1]:
# 1) Инсталлируем пакеты
!pip -q install "langchain>=0.2.11" "langchain-community>=0.2.10" "langchain-openai>=0.1.7" \
                "langchain-huggingface>=0.1.0" \
                "faiss-cpu>=1.8.0" "sentence-transformers>=3.0.1" "tiktoken>=0.7.0" \
                "duckduckgo-search>=5.3.1" "trafilatura>=1.8.0" "gradio>=4.36.1" \
                "python-docx>=1.1.0" "pypdf>=4.2.0" "rank-bm25>=0.2.2" "requests>=2.31.0"

# =========================== КОД ПРОГРАММЫ ===========================
import os, re, time, shutil, json, math, requests, zipfile, random
from datetime import datetime, timezone
from typing import List, Dict, Tuple, Any
from urllib.parse import urlparse

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEndpoint

from duckduckgo_search import DDGS
import trafilatura

import gradio as gr
from docx import Document as DocxDocument
from pypdf import PdfReader
from docx import Document as DocxReader

try:
    from sentence_transformers import CrossEncoder
    _HAS_CE = True
except Exception:
    _HAS_CE = False

from rank_bm25 import BM25Okapi
import numpy as np

random.seed(42)

# ------------------ Утилиты ------------------
def parse_date_str(s: str):
    try:
        if not s: return None
        return datetime.fromisoformat(s).replace(tzinfo=timezone.utc)
    except Exception:
        try: return datetime.strptime(s[:10], "%Y-%m-%d").replace(tzinfo=timezone.utc)
        except Exception: return None

def recency_boost(date_str: str, tau_days: int = 90) -> float:
    dt = parse_date_str(date_str)
    if not dt: return 0.5
    age_days = max(0.0, (datetime.now(timezone.utc)-dt).total_seconds()/86400.0)
    return math.exp(-age_days/max(1.0, float(tau_days)))

RELIABLE = {
    "openai.com": 1.0, "deepmind.com": 1.0, "arxiv.org": 1.0,
    "huggingface.co": 0.95, "nvidia.com": 0.95, "anthropic.com": 0.95,
    "technologyreview.com": 0.95,
}
BLOCKED_DOMAINS = {"rutube.ru","vk.com","ok.ru","tiktok.com","x.com","twitter.com","youtube.com","youtu.be"}

# ---- Расширенный curated-пул RU-AI ----
RUS_AI_SITES = {
    # Медиа/каталоги
    "tadviser.ru",

    # Университеты и центры
    "ai.hse.ru", "www.hse.ru", "cs.hse.ru",
    "skoltech.ru", "crei.skoltech.ru",
    "mipt.ru", "ai.mipt.ru", "phystech.edu",
    "msu.ru", "vmk.msu.ru", "ai.msu.ru", "ai-institute.ru",
    "innopolis.ru", "university.innopolis.ru",
    "itmo.ru", "news.itmo.ru", "ai.itmo.ru",
    "spbstu.ru", "news.spbstu.ru",
    "nsu.ru", "sfedu.ru", "urfu.ru", "bmstu.ru", "mai.ru",

    # Институты РАН и окрестности
    "ispras.ru",   # ИСП РАН
    "iitp.ru",     # ИПИ РАН
    "isa.ru", "ipiran.ru", "ras.ru",

    # Индустрия / исследовательские центры компаний
    "airi.net",    # AIRI
    "sber.ai", "sberbank.ru", "sberdevices.ru",
    "yandex.ru", "yandexdataschool.ru", "yandex.com/company/research",
    "gazprom-neft.ru",
}

# Повышающие веса для RU-AI источников
RELIABLE.update({
    "tadviser.ru": 0.98,
    "ai.hse.ru": 0.98, "hse.ru": 0.96, "cs.hse.ru": 0.96,
    "skoltech.ru": 0.97, "crei.skoltech.ru": 0.97,
    "mipt.ru": 0.96, "ai.mipt.ru": 0.98, "phystech.edu": 0.95,
    "msu.ru": 0.95, "vmk.msu.ru": 0.96, "ai.msu.ru": 0.97, "ai-institute.ru": 0.97,
    "innopolis.ru": 0.95, "university.innopolis.ru": 0.96,
    "itmo.ru": 0.96, "news.itmo.ru": 0.96, "ai.itmo.ru": 0.97,
    "spbstu.ru": 0.94, "news.spbstu.ru": 0.94,
    "nsu.ru": 0.94, "sfedu.ru": 0.93, "urfu.ru": 0.92, "bmstu.ru": 0.92, "mai.ru": 0.92,
    "ispras.ru": 0.98, "iitp.ru": 0.97, "isa.ru": 0.95, "ipiran.ru": 0.95, "ras.ru": 0.93,
    "airi.net": 0.98,
    "sber.ai": 0.95, "sberbank.ru": 0.93, "sberdevices.ru": 0.94,
    "yandex.ru": 0.94, "yandexdataschool.ru": 0.96, "yandex.com": 0.93,
    "gazprom-neft.ru": 0.9,
})

def source_weight(url: str) -> float:
    try:
        if url.startswith("file://"): return 0.95
        host = urlparse(url).netloc
        if host in BLOCKED_DOMAINS: return 0.5
        for k,v in RELIABLE.items():
            if k in host: return v
        return 0.9
    except Exception:
        return 0.9

def tokenize(text: str):
    text = text.lower()
    text = re.sub(r"[^a-zа-я0-9\s]", " ", text)
    return [w for w in text.split() if len(w) > 2]

# ---- Безопасные цитаты ----
SENT_END = re.compile(r"[.!?…]+[\)\]»”]*\s+")
def _clean_ws(s: str) -> str:
    return re.sub(r"\s+", " ", (s or "").strip())
def _smart_clip(s: str, max_len: int) -> str:
    s = _clean_ws(s)
    if len(s) <= max_len: return s
    cut = s[:max_len]
    m = list(SENT_END.finditer(cut))
    if m: return _clean_ws(s[:m[-1].end()]).rstrip()
    last_space = cut.rfind(" ")
    if last_space > max_len * 0.6: return _clean_ws(cut[:last_space]) + "…"
    return _clean_ws(cut) + "…"
def best_quote(text: str, query: str, max_len: int = 280) -> str:
    sents = re.split(r"(?<=[.!?…])\s+", (text or "").strip())
    if not sents: return _smart_clip(text or "", max_len)
    q = set(tokenize(query))
    scored = []
    for i, s in enumerate(sents):
        toks = set(tokenize(s))
        scored.append((len(q & toks), i))
    scored.sort(reverse=True)
    if scored and scored[0][0] > 0:
        i = scored[0][1]
        window = " ".join(sents[max(0, i-1): i+2])
    else:
        window = " ".join(sents[:2])
    return _smart_clip(window, max_len)

def normalize_dense_score(distance: float) -> float:
    return 1.0 / (1.0 + max(0.0, float(distance)))

# ---- Санитайзер запроса ----
def sanitize_query(q: str) -> str:
    if not q: return q
    s = q.strip()
    if "Вопрос:" in s: s = s.split("Вопрос:")[-1]
    s = re.sub(r"(?im)^\s*(ты|you)\s*[—-]\s.*?$", "", s)
    for line in s.splitlines():
        if "?" in line:
            s = line.strip()
            break
    return s.strip(" \"'`")

# ------------------ Seed корпус (ИИ) ------------------
SEED_CORPUS = [
    {"title":"Мультимодальные модели и агентные системы","url":"local://multimodal_agents","date":"2025-03-15",
     "text":"""Мультимодальные LLM объединяют текст, изображение и видео, повышая способность к комплексным задачам.
Агентные надстройки позволяют моделям планировать шаги и использовать инструменты. Важен мониторинг фактов и ограничение галлюцинаций."""},
    {"title":"Энергоэффективность и оптимизация инференса","url":"local://efficiency_inference","date":"2025-02-10",
     "text":"""Тренд на энергоэффективность: квантование, sparsity, дистилляция, компиляторы графов (TensorRT/TVM).
Это снижает задержку и стоимость инференса на CPU/GPU/NPUs, а также облегчает деплой на edge."""},
    {"title":"MLOps: качество данных, наблюдаемость и стоимость","url":"local://mlops_data_quality","date":"2025-01-20",
     "text":"""Ключевые темы MLOps: качество данных (data curation), наблюдаемость (observability), метрики стоимости и guardrails.
Практики: слежение за дрейфом, эвалюации по задачам, журналирование промптов, безопасная оркестрация инструментов."""},
    {"title":"Регулирование ИИ и риск-менеджмент","url":"local://ai_governance","date":"2025-04-05",
     "text":"""Усиливается регулирование ИИ и требования к управлению рисками. Организации внедряют политику, карточки моделей,
процедуры тестирования безопасности, отслеживание источников данных и аудит цепочки поставок ML-артефактов."""},
    {"title":"Инфраструктура: память и межсоединения","url":"local://infra_memory_interconnect","date":"2025-02-28",
     "text":"""Рост параметров и контекстных окон повышает требования к памяти и пропускной способности межсоединений (NVLink/PCIe).
Оптимизация планировщиков, offloading и sharding помогают утилизировать кластеры эффективнее."""},
    {"title":"Прикладные кейсы: код, дизайн, документы","url":"local://apps_productivity","date":"2025-03-01",
     "text":"""Генеративный ИИ внедряется в продактивити-инструменты: помощники по коду, дизайну, резюмированию документов.
Растёт спрос на надёжные цитаты, ссылки и контроль фактов (RAG с временными фильтрами и reranking)."""},
]

# ------------------ Пути и глобалы ------------------
BASE_INDEX_DIR = "/content/faiss_index_base"
USER_INDEX_DIR = "/content/faiss_index_user"
AI_INDEX_DIR   = "/content/faiss_index_ai"

BASE_CHUNKS_JSON = "/content/chunks_base.jsonl"
USER_CHUNKS_JSON = "/content/chunks_user.jsonl"
AI_CHUNKS_JSON   = "/content/chunks_ai.jsonl"

WEB_CACHE  = "/content/web_cache.jsonl"
USER_CACHE = "/content/user_docs.jsonl"
AI_CACHE   = "/content/ai_pool.jsonl"

DOWNLOAD_DIR = "/content/downloaded_sources"

# Включать ли авто-бутстрап при старте (для пула ИИ)
RUN_AI_POOL_BOOTSTRAP = False  # по умолчанию выкл.

# Эмбеддинги
EMB_NAME_PRIMARY = "intfloat/multilingual-e5-small"
EMB_NAME_FALLBACK = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

def build_embeddings():
    try:
        emb = HuggingFaceEmbeddings(model_name=EMB_NAME_PRIMARY)
        _ = emb.embed_query("warmup")
        return emb
    except Exception as e:
        print("⚠️ Не удалось загрузить", EMB_NAME_PRIMARY, "— fallback на", EMB_NAME_FALLBACK, "| Ошибка:", e)
        emb = HuggingFaceEmbeddings(model_name=EMB_NAME_FALLBACK)
        _ = emb.embed_query("warmup")
        return emb

emb = build_embeddings()
splitter = RecursiveCharacterTextSplitter(
    chunk_size=900, chunk_overlap=180, separators=["\n\n", "\n", ". ", "。", "！", "？"]
)

# BM25 в памяти
BASE_BM25 = {"model":None, "metas":[], "texts":{}, "id_by_chunk":{}}
USER_BM25 = {"model":None, "metas":[], "texts":{}, "id_by_chunk":{}}
AI_BM25   = {"model":None, "metas":[], "texts":{}, "id_by_chunk":{}}
cross_encoder = None  # опционально

# ------------------ Индексация + BM25 ------------------
def docs_from_corpus(corpus: List[Dict]) -> List[Document]:
    return [Document(page_content=a["text"], metadata={"title":a["title"],"url":a["url"],"date":a["date"]}) for a in corpus]

def write_chunks_jsonl(chunks: List[Document], jsonl_path: str):
    with open(jsonl_path, "w", encoding="utf-8") as f:
        for ch in chunks:
            meta = {**ch.metadata}
            f.write(json.dumps({"chunk_id": meta["chunk_id"], "text": ch.page_content, "meta": meta}, ensure_ascii=False)+"\n")

def build_bm25_from_jsonl(jsonl_path: str):
    tokenized, metas, texts, id_by_chunk = [], [], {}, {}
    if not os.path.exists(jsonl_path):
        return None, [], {}, {}
    with open(jsonl_path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            obj = json.loads(line)
            chunk_id = obj["chunk_id"]; text = obj["text"]; meta = obj["meta"]
            tokenized.append(tokenize(text)); metas.append(meta); texts[chunk_id] = text; id_by_chunk[chunk_id] = i
    model = BM25Okapi(tokenized) if tokenized else None
    return model, metas, texts, id_by_chunk

def build_index_with_bm25(initial_docs: List[Document], index_dir: str, chunks_json: str, target_store: dict):
    if os.path.exists(index_dir): shutil.rmtree(index_dir)
    chunks_raw = splitter.split_documents(initial_docs)
    chunks = []
    for i, ch in enumerate(chunks_raw):
        meta = {**ch.metadata, "chunk_id": i}
        chunks.append(Document(page_content=ch.page_content, metadata=meta))
    if not chunks:
        # нет документов — обнуляем хранилище
        target_store.update({"model":None, "metas":[], "texts":{}, "id_by_chunk":{}})
        if os.path.exists(chunks_json): os.remove(chunks_json)
        if os.path.exists(index_dir): shutil.rmtree(index_dir)
        return
    write_chunks_jsonl(chunks, chunks_json)
    vectordb = FAISS.from_documents(chunks, emb); vectordb.save_local(index_dir)
    model, metas, texts, id_by_chunk = build_bm25_from_jsonl(chunks_json)
    target_store["model"] = model; target_store["metas"] = metas; target_store["texts"] = texts; target_store["id_by_chunk"] = id_by_chunk

def load_index(index_dir: str) -> FAISS | None:
    try:
        if not os.path.exists(index_dir): return None
        return FAISS.load_local(index_dir, emb, allow_dangerous_deserialization=True)
    except Exception:
        return None

# первичная сборка base
build_index_with_bm25(docs_from_corpus(SEED_CORPUS), BASE_INDEX_DIR, BASE_CHUNKS_JSON, BASE_BM25)

# если существует пул ИИ — прогружаем его
if os.path.exists(AI_CACHE):
    rows_ai = []
    with open(AI_CACHE, "r", encoding="utf-8") as f:
        for line in f:
            try: rows_ai.append(json.loads(line))
            except: pass
    if rows_ai:
        build_index_with_bm25(docs_from_corpus(rows_ai), AI_INDEX_DIR, AI_CHUNKS_JSON, AI_BM25)

# ------------------ Веб-обогащение (для базового индекса) ------------------
HDRS = {"User-Agent":"Mozilla/5.0 (Colab-RAG/1.0)"}
def is_blocked(url: str) -> bool:
    host = urlparse(url).netloc
    return host in BLOCKED_DOMAINS

def fetch_text_with_timeout(url: str, timeout=6) -> str:
    try:
        r = requests.get(url, headers=HDRS, timeout=timeout, allow_redirects=True)
        if r.status_code>=400: return ""
        if is_blocked(url): return ""
        html = r.text
        txt = trafilatura.extract(html, include_comments=False, include_images=False) or ""
        return (txt or "").strip()
    except Exception:
        return ""

def fetch_pages_duckduckgo(query: str, max_pages: int = 2) -> List[Dict]:
    results = []
    with DDGS() as ddgs:
        for r in ddgs.text(query, region="wt-wt", safesearch="moderate", max_results=max_pages):
            url = r.get("href") or r.get("url"); title = (r.get("title") or "")[:160]
            if not url or is_blocked(url): continue
            text = fetch_text_with_timeout(url, timeout=6)
            if len(text) < 800: continue
            results.append({"title": title if title else url, "url": url,
                            "date": datetime.utcnow().strftime("%Y-%m-%d"),
                            "text": text[:100_000]})
    return results

def read_jsonl(path: str) -> List[Dict]:
    data = []
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                try: data.append(json.loads(line))
                except Exception: pass
    return data

def write_jsonl(path: str, rows: List[Dict]):
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False)+"\n")

def enrich_base_index_with_web(query: str, max_pages: int = 2):
    new_pages = fetch_pages_duckduckgo(query, max_pages=max_pages)
    if not new_pages: return 0
    existing = read_jsonl(WEB_CACHE)
    by_url = {p["url"]: p for p in existing}
    for p in new_pages: by_url[p["url"]] = p
    merged = list(by_url.values())
    write_jsonl(WEB_CACHE, merged)
    full_corpus = SEED_CORPUS + merged
    build_index_with_bm25(docs_from_corpus(full_corpus), BASE_INDEX_DIR, BASE_CHUNKS_JSON, BASE_BM25)
    return len(new_pages)

# ---- Обогащение по RU-AI (для базового индекса) ----
def fetch_pages_duckduckgo_ru_ai(query: str, max_pages: int = 3) -> List[Dict]:
    results = []
    with DDGS() as ddgs:
        for dom in list(RUS_AI_SITES):
            q = f"site:{dom} {query}"
            for r in ddgs.text(q, region="ru-ru", safesearch="moderate", max_results=1):  # 1/домен
                url = r.get("href") or r.get("url")
                title = (r.get("title") or "")[:160]
                if not url or is_blocked(url): continue
                text = fetch_text_with_timeout(url, timeout=6)
                if len(text) < 800: continue
                results.append({"title": title or url, "url": url,
                                "date": datetime.utcnow().strftime("%Y-%m-%d"),
                                "text": text[:100_000]})
        # общий добор
        for r in ddgs.text(query, region="ru-ru", safesearch="moderate", max_results=max_pages):
            url = r.get("href") or r.get("url")
            title = (r.get("title") or "")[:160]
            if not url or is_blocked(url): continue
            text = fetch_text_with_timeout(url, timeout=6)
            if len(text) < 800: continue
            results.append({"title": title or url, "url": url,
                            "date": datetime.utcnow().strftime("%Y-%m-%d"),
                            "text": text[:100_000]})
    return results

def enrich_base_index_with_ru_ai(query: str, max_pages: int = 3):
    new_pages = fetch_pages_duckduckgo_ru_ai(query, max_pages=max_pages)
    if not new_pages: return 0
    existing = read_jsonl(WEB_CACHE)
    by_url = {p["url"]: p for p in existing}
    for p in new_pages: by_url[p["url"]] = p
    merged = list(by_url.values())
    write_jsonl(WEB_CACHE, merged)
    full_corpus = SEED_CORPUS + merged
    build_index_with_bm25(docs_from_corpus(full_corpus), BASE_INDEX_DIR, BASE_CHUNKS_JSON, BASE_BM25)
    return len(new_pages)

# ---- Пул источников про ИИ (отдельный индекс + URL-добавление) ----
RU_AI_BOOTSTRAP_QUERIES = [
    "искусственный интеллект","машинное обучение","нейросети",
    "генеративный ИИ","LLM","RAG","агентные системы"
]

def rebuild_ai_index(rows: List[Dict]) -> int:
    if not rows:
        if os.path.exists(AI_INDEX_DIR): shutil.rmtree(AI_INDEX_DIR)
        AI_BM25.update({"model":None, "metas":[], "texts":{}, "id_by_chunk":{}})
        if os.path.exists(AI_CHUNKS_JSON): os.remove(AI_CHUNKS_JSON)
        return 0
    build_index_with_bm25(docs_from_corpus(rows), AI_INDEX_DIR, AI_CHUNKS_JSON, AI_BM25)
    return len(rows)

def add_pages_to_ai_pool(pages: List[Dict]) -> Tuple[int, int]:
    if not pages: return 0, 0
    existing = read_jsonl(AI_CACHE)
    before = len(existing)
    by_url = {p["url"]: p for p in existing}
    for p in pages: by_url[p["url"]] = p
    merged = list(by_url.values())
    write_jsonl(AI_CACHE, merged)
    rebuilt = rebuild_ai_index(merged)
    return len(merged) - before, rebuilt

def add_urls_to_ai_pool(urls: List[str], timeout:int=8) -> Tuple[int,int]:
    pages = []
    for u in urls:
        u = (u or "").strip()
        if not u or not (u.startswith("http://") or u.startswith("https://")): continue
        if is_blocked(u): continue
        txt = fetch_text_with_timeout(u, timeout=timeout)
        if len(txt) < 800: continue
        pages.append({
            "title": u,
            "url": u,
            "date": datetime.utcnow().strftime("%Y-%m-%d"),
            "text": txt[:100_000]
        })
    return add_pages_to_ai_pool(pages)

# ------------------ Пользовательские документы ------------------
def _extract_path_from_gradio_file(f: Any) -> str:
    if f is None: return ""
    if hasattr(f, "name"): return f.name
    if isinstance(f, (str, bytes)): return f if isinstance(f, str) else f.decode("utf-8", "ignore")
    if isinstance(f, dict): return f.get("name") or f.get("path") or f.get("orig_name") or ""
    return str(f)

def read_txt(path: str) -> str:
    try:
        with open(path, "r", encoding="utf-8", errors="ignore") as f: return f.read()
    except Exception: return ""
def read_md(path: str) -> str: return read_txt(path)
def read_html(path: str) -> str:
    try:
        with open(path, "r", encoding="utf-8", errors="ignore") as fh:
            html = fh.read()
        return trafilatura.extract(html, include_comments=False, include_images=False) or ""
    except Exception: return ""
def read_pdf(path: str, max_pages: int = 40) -> str:
    try:
        reader = PdfReader(path)
        parts = []
        for i, page in enumerate(reader.pages):
            if i >= max_pages: break
            t = page.extract_text() or ""
            t = re.sub(r"-\s*\n", "", t)
            t = re.sub(r"\n+", "\n", t)
            parts.append(t)
        return "\n".join(parts)
    except Exception: return ""
def read_docx(path: str) -> str:
    try:
        doc = DocxReader(path)
        return "\n".join([p.text for p in doc.paragraphs])
    except Exception: return ""

EXT_READERS = {".txt":read_txt, ".md":read_md, ".html":read_html, ".htm":read_html, ".pdf":read_pdf, ".docx":read_docx}

def load_user_files(files: List) -> Tuple[int, List[Dict]]:
    rows, ok = [], 0
    for f in (files or []):
        try:
            path = _extract_path_from_gradio_file(f)
            if not path or not os.path.exists(path): continue
            ext = os.path.splitext(path)[1].lower()
            reader = EXT_READERS.get(ext)
            if not reader: continue
            text = (reader(path) or "").strip()
            if len(text) < 200: continue
            rows.append({"title": os.path.basename(path), "url": f"file://{os.path.basename(path)}",
                         "date": datetime.utcnow().strftime("%Y-%m-%d"), "text": text[:150_000]})
            ok += 1
        except Exception:
            continue
    return ok, rows

def rebuild_user_index(rows: List[Dict]) -> int:
    if not rows:
        if os.path.exists(USER_INDEX_DIR): shutil.rmtree(USER_INDEX_DIR)
        USER_BM25.update({"model":None, "metas":[], "texts":{}, "id_by_chunk":{}})
        return 0
    build_index_with_bm25(docs_from_corpus(rows), USER_INDEX_DIR, USER_CHUNKS_JSON, USER_BM25)
    return len(rows)

def clear_user_index():
    if os.path.exists(USER_INDEX_DIR): shutil.rmtree(USER_INDEX_DIR)
    if os.path.exists(USER_CACHE): os.remove(USER_CACHE)
    USER_BM25.update({"model":None, "metas":[], "texts":{}, "id_by_chunk":{}})

# ------------------ Гибридный поиск ------------------
def get_store(index_key: str):
    if index_key == "user": return USER_BM25, USER_INDEX_DIR, USER_CHUNKS_JSON
    if index_key == "ai":   return AI_BM25, AI_INDEX_DIR, AI_CHUNKS_JSON
    return BASE_BM25, BASE_INDEX_DIR, BASE_CHUNKS_JSON

def hybrid_search(query: str, index_key: str, k_dense=30, k_sparse=30, k_final=5,
                  freshness_lambda=0.3, max_age_months=12, min_final_score=0.25):
    store, index_dir, _ = get_store(index_key)
    vectordb = load_index(index_dir)
    bm25 = store["model"]; metas = store["metas"]; texts = store["texts"]
    if bm25 is None or vectordb is None: return []

    queries = [query]  # без искусственного "AI"-сдвига

    candidates = {}
    for q in queries:
        try:
            dense_hits = vectordb.similarity_search_with_score(q, k=k_dense)
        except Exception:
            dense_hits = []
        for doc, dist in dense_hits:
            cid = doc.metadata.get("chunk_id")
            if cid is None: continue
            entry = candidates.get(cid, {"meta": doc.metadata, "dense":0.0, "sparse":0.0})
            entry["dense"] = max(entry["dense"], normalize_dense_score(dist))
            candidates[cid] = entry

        tokens = tokenize(q)
        scores = bm25.get_scores(tokens)
        idx = np.argsort(scores)[::-1][:k_sparse]
        for i in idx:
            meta = metas[i]
            cid = meta.get("chunk_id")
            sc = float(scores[i])
            entry = candidates.get(cid, {"meta": meta, "dense":0.0, "sparse":0.0})
            entry["sparse"] = max(entry["sparse"], sc)
            candidates[cid] = entry

    if not candidates: return []

    s_vals = [c["sparse"] for c in candidates.values()]
    s_min, s_max = min(s_vals), max(s_vals)
    def norm_s(x): return 0.0 if s_max==s_min else (x - s_min)/(s_max - s_min)

    ranked = []
    for cid, c in candidates.items():
        meta = c["meta"]
        if max_age_months and isinstance(max_age_months, (int,float)) and max_age_months>0:
            dt = parse_date_str(meta.get("date",""))
            if dt:
                age_months = (datetime.now(timezone.utc) - dt).days / 30.0
                if age_months > max_age_months:
                    continue
        dense_n = float(c["dense"])
        sparse_n = norm_s(float(c["sparse"]))
        base = 0.6*dense_n + 0.4*sparse_n
        fres = recency_boost(meta.get("date",""))
        relw = source_weight(meta.get("url",""))
        final = base * (1.0 + float(freshness_lambda)*fres) * relw
        final *= (0.99 + 0.02*random.random())
        if final >= float(min_final_score):
            ranked.append({"chunk_id": cid, "meta": meta, "dense": dense_n, "sparse": sparse_n,
                           "fresh": fres, "relw": relw, "final": final})

    ranked.sort(key=lambda x: x["final"], reverse=True)
    return ranked[:k_final]

# ------------------ LLM провайдеры ------------------
LLM_CFG = {"provider":"none", "model":"", "base_url":"", "api_key":"", "hf_token":""}
LLM_OBJ = None
last_sources_store: List[Dict] = []

def apply_llm_settings(provider:str, model:str, api_key:str, base_url:str, hf_token:str) -> str:
    global LLM_CFG, LLM_OBJ
    LLM_CFG.update({"provider":provider, "model":model, "base_url":base_url, "api_key":api_key, "hf_token":hf_token})
    LLM_OBJ = None

    if provider == "none":
        return "✅ Строгий режим без LLM: ответы только из источников."

    if provider == "openai_compatible":
        if not api_key:
            return "⚠️ Укажите API Key."
        os.environ["OPENAI_API_KEY"] = api_key
        if base_url: os.environ["OPENAI_BASE_URL"] = base_url
        else: os.environ.pop("OPENAI_BASE_URL", None)
        m = model or "gpt-4o-mini"
        try:
            LLM_OBJ = ChatOpenAI(model=m, temperature=0.2)
            _ = LLM_OBJ.invoke("ping").content
            return f"✅ Готово: OpenAI-совместимый LLM '{m}'."
        except Exception as e:
            LLM_OBJ = None
            return f"⚠️ Не удалось инициализировать OpenAI-совместимый LLM: {e}"

    if provider == "hf_inference":
        if not hf_token or not model:
            return "⚠️ Укажите HF Token и модель (например, HuggingFaceH4/zephyr-7b-beta)."
        os.environ["HUGGINGFACEHUB_API_TOKEN"] = hf_token
        try:
            LLM_OBJ = HuggingFaceEndpoint(
                repo_id=model,
                task="text-generation",
                temperature=0.2,
                max_new_tokens=512,
                do_sample=False,
                return_full_text=False,
            )
            _ = LLM_OBJ.invoke("ping")
            return f"✅ Готово: HuggingFace Inference '{model}'."
        except Exception as e:
            LLM_OBJ = None
            return f"⚠️ Не удалось инициализировать HF Inference: {e}"

    return "⚠️ Неизвестный провайдер."

# ------------------ Ответы ------------------
def render_bullets_with_highlights(items: List[Tuple[str, Dict]]) -> str:
    out = []
    for quote, meta in items:
        out.append(f'> **"{quote}"**\n\n— [{meta.get("title")}]({meta.get("url")}) ({meta.get("date")})')
    return "\n\n".join(out)

def gen_answer_strict(query: str, hits: List[Dict], texts: Dict[int,str]) -> str:
    if not hits: return "Не найдено поддержанных источников по вашему запросу."
    items = []
    for h in hits[:4]:
        meta = h["meta"]; text = texts.get(h["chunk_id"], "")
        quote = best_quote(text, query)
        items.append((quote, meta))
    return "Ключевые тезисы (строго из источников):\n\n" + render_bullets_with_highlights(items)

def gen_answer_with_llm(query: str, hits: List[Dict], texts: Dict[int,str]) -> str:
    if not hits: return "Не найдено поддержанных источников."
    if LLM_CFG["provider"] == "none" or LLM_OBJ is None:
        return gen_answer_strict(query, hits, texts)

    blocks = []
    for i, h in enumerate(hits, 1):
        meta = h["meta"]; text = texts.get(h["chunk_id"], "")
        blocks.append(f"[{i}] {meta.get('title')} | {meta.get('url')} | {meta.get('date')}\n{text}\n---")

    prompt_system = ("Ты — аналитик. Отвечай строго на основе данных ниже. "
                     "2–5 тезисов; короткие цитаты в кавычках; после ответа — список источников с номерами; не выдумывай.")
    prompt_user = f"Вопрос: {query}\n\n[ДОКУМЕНТЫ]\n" + "\n".join(blocks) + "\n\nОтвет:"

    try:
        if isinstance(LLM_OBJ, ChatOpenAI):
            from langchain_core.messages import SystemMessage, HumanMessage
            resp = LLM_OBJ.invoke([SystemMessage(content=prompt_system), HumanMessage(content=prompt_user)])
            return resp.content
        else:
            full_prompt = prompt_system + "\n\n" + prompt_user
            resp = LLM_OBJ.invoke(full_prompt)
            return str(resp)
    except Exception as e:
        return f"⚠️ LLM не ответил ({e}). Перехожу на строгий режим.\n\n" + gen_answer_strict(query, hits, texts)

AI_KEYWORDS = {
    "ai","ии","искусственный интеллект","ml","машинное обучение",
    "llm","нейросеть","нейросети","генератив", "rag","агент", "mlops",
    "prompt","промпт","модель","датасет","fine tune","дообучен","дообучение"
}
def is_ai_topic(query: str) -> bool:
    t = tokenize(query)
    joined = " ".join(t)
    return any(k in joined for k in AI_KEYWORDS)
def _is_russian(s: str) -> bool:
    return bool(re.search(r"[а-яА-ЯёЁ]", s or ""))

def run_rag(user_raw_query: str, only_user: bool, only_ai_pool: bool, use_web_toggle: bool, top_k: int,
            freshness_lambda: float, max_age_months: int, min_final_score: float):
    q = sanitize_query(user_raw_query)

    # Веб-добор только для базового индекса
    do_web = (not only_user) and (not only_ai_pool) and bool(use_web_toggle)
    if do_web:
        if _is_russian(q) and is_ai_topic(q):
            _ = enrich_base_index_with_ru_ai(q, max_pages=3)
        else:
            _ = enrich_base_index_with_web(q, max_pages=2)

    # Выбор индекса
    if only_user:
        index_key = "user"; store = USER_BM25
    elif only_ai_pool:
        index_key = "ai"; store = AI_BM25
    else:
        index_key = "base"; store = BASE_BM25

    hits = hybrid_search(
        q,
        index_key=index_key,
        k_dense=30,
        k_sparse=30,
        k_final=top_k,
        freshness_lambda=freshness_lambda,
        max_age_months=max_age_months,
        min_final_score=min_final_score,
    )
    texts = store["texts"]

    if not hits:
        if only_ai_pool:
            return ("В пуле про ИИ недостаточно релевантных источников для ответа. "
                    "Нажмите «Собрать пул про ИИ» или добавьте URL.", [])
        return ("Недостаточно релевантных источников для ответа. "
                "Включите опцию «Дополнить инфо из Интернета» или уточните запрос.", [])

    answer = gen_answer_strict(q, hits, texts) if (LLM_CFG["provider"] == "none" or LLM_OBJ is None) \
             else gen_answer_with_llm(q, hits, texts)

    # источники (дедуп по домену)
    seen_hosts, seen_urls, sources = set(), set(), []
    for h in hits:
        url = h["meta"].get("url","")
        host = urlparse(url).netloc
        if host in seen_hosts or url in seen_urls: continue
        seen_hosts.add(host); seen_urls.add(url)
        sources.append({"title": h["meta"].get("title"), "url": url, "date": h["meta"].get("date")})
    return answer, sources

# ------------------ DOCX отчёт ------------------
def save_docx_report(history_msgs: List[Dict]) -> str:
    path = "/content/RAG_Assistant_Report.docx"
    doc = DocxDocument()
    doc.add_heading("RAG-Ассистент — Отчёт", level=1)
    doc.add_paragraph(f"Дата: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    q_idx = 0
    for i in range(0, len(history_msgs), 2):
        if i < len(history_msgs) and history_msgs[i].get("role")=="user":
            q_idx += 1
            q = history_msgs[i]["content"]
            doc.add_heading(f"Q{q_idx}: {q}", level=2)
            if i+1 < len(history_msgs) and history_msgs[i+1].get("role")=="assistant":
                a = history_msgs[i+1]["content"]
                doc.add_paragraph(a)
    doc.save(path)
    return path

# ------------------ Скачивание использованных источников ------------------
def safe_filename_from_url(url: str) -> str:
    try:
        u = urlparse(url)
        name = (u.netloc + u.path).strip("/")
        name = re.sub(r"[^a-zA-Z0-9._-]", "_", name)
        return name or "source"
    except Exception:
        return "source"

def download_sources_zip(sources: List[Dict]) -> str:
    if not sources: return ""
    os.makedirs(DOWNLOAD_DIR, exist_ok=True)
    saved_files = []
    for s in sources:
        url = s.get("url") or ""
        if not url or url.startswith(("local://","file://")): continue
        host = urlparse(url).netloc
        if host in BLOCKED_DOMAINS: continue
        try:
            r = requests.get(url, headers={"User-Agent":"Mozilla/5.0 (Colab-RAG/1.0)"}, timeout=12, allow_redirects=True)
            if r.status_code >= 400: continue
            fname = safe_filename_from_url(url) + ".html"
            fpath = os.path.join(DOWNLOAD_DIR, fname)
            with open(fpath, "w", encoding="utf-8") as f:
                f.write(r.text)
            saved_files.append(fpath)
        except Exception:
            continue
    if not saved_files: return ""
    zip_path = "/content/used_sources.zip"
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for fp in saved_files:
            zf.write(fp, arcname=os.path.basename(fp))
    return zip_path

# ------------------ UI (Gradio) ------------------
LLM_CFG = {"provider":"none", "model":"", "base_url":"", "api_key":"", "hf_token":""}
LLM_OBJ = None
chat_history_store = []   # [{"role": "user"/"assistant", "content": "..."}]
last_sources_store = []
_last_q = {"text": None, "answer": None}

with gr.Blocks(title="RAG — универсальный Q&A") as demo:
    gr.Markdown("## RAG-ассистент: отвечает на любые вопросы\nПо умолчанию — универсальный режим. Для задач про ИИ можно собрать и использовать отдельный пул источников.")

    with gr.Accordion("⚙️ Настройки LLM (по желанию)", open=False):
        provider = gr.Dropdown(choices=["none","openai_compatible","hf_inference"], value="none", label="LLM провайдер")
        llm_model = gr.Textbox(value="gpt-4o-mini", label="Модель (OpenAI-совм.) / repo_id (HF Inference)")
        llm_api_key = gr.Textbox(value="", type="password", label="API Key (OpenAI/OpenRouter)")
        llm_base_url = gr.Textbox(value="", label="Base URL (опц., напр. OpenRouter: https://openrouter.ai/api/v1)")
        hf_token = gr.Textbox(value="", type="password", label="HF Token (для HuggingFace Inference)")
        apply_btn = gr.Button("Применить LLM настройки")
        llm_status = gr.Markdown("Текущий режим: **строгий (без LLM)**.")

    with gr.Row():
        web_toggle = gr.Checkbox(label="Дополнить инфо из Интернета", value=False)
        only_user_toggle = gr.Checkbox(label="Отвечать только по моим документам", value=False)
        only_ai_toggle = gr.Checkbox(label="Только на основе пула про ИИ", value=False)
        topk = gr.Slider(3, 10, value=5, step=1, label="Top-K документов")
    with gr.Row():
        freshness_lambda = gr.Slider(0.0, 1.0, value=0.3, step=0.05, label="Вес свежести (λ)")
        max_age_months = gr.Slider(0, 36, value=12, step=1, label="Макс. возраст источников (мес, 0=без ограничения)")
        min_final_score = gr.Slider(0.0, 1.0, value=0.25, step=0.05, label="Порог релевантности (min score)")

    gr.Markdown("### Пул источников про ИИ")
    with gr.Row():
        bootstrap_btn = gr.Button("🚀 Собрать пул про ИИ", variant="secondary")
        add_url_text = gr.Textbox(placeholder="Вставьте один или несколько URL (по одному на строке)", label="Добавить URL")
        add_url_btn = gr.Button("Добавить URL", variant="primary")
    ai_pool_status = gr.Markdown("Пул про ИИ пока пуст или не обновлялся.")

    gr.Markdown("### Загрузка файлов (PDF, DOCX, TXT, MD, HTML)")
    with gr.Row():
        file_uploader = gr.File(file_count="multiple", file_types=[".pdf",".docx",".txt",".md",".html",".htm"], label="Файлы")
    with gr.Row():
        add_btn = gr.Button("➕ Добавить в индекс (мои документы)", variant="primary")
        clear_docs_btn = gr.Button("🧹 Очистить мои документы")

    status_md = gr.Markdown("Готов к работе.")
    chatbot = gr.Chatbot(type="messages", height=420, render_markdown=True, show_copy_button=True)
    msg = gr.Textbox(placeholder="Спросите что угодно. Пример: Как выбрать ноутбук для ML в 2025?", label="Ваш вопрос")

    with gr.Row():
        send_btn = gr.Button("Отправить", variant="primary")
        stop_btn = gr.Button("⏹️ Стоп")
        clear_chat_btn = gr.Button("🧽 Очистить чат")
        report_btn = gr.Button("Скачать отчёт (.docx)")
        download_btn = gr.Button("Скачать использованные источники (ZIP)")
    report_file = gr.File(label="Отчёт", visible=False)
    sources_file = gr.File(label="Источники (ZIP)", visible=False)

    # --- Handlers ---
    def on_apply_llm(p, m, k, base, hf):
        st = apply_llm_settings(p, m, k, base, hf)
        return f"{st}\nТекущий провайдер: **{LLM_CFG['provider']}**; модель: **{LLM_CFG['model'] or '—'}**."

    def on_add(files):
        t0 = time.time()
        ok, rows = load_user_files(files)
        if ok == 0:
            return "⚠️ Не удалось прочитать файлы. Поддерживаемые типы: PDF, DOCX, TXT, MD, HTML.", gr.update(value=[])
        existing = read_jsonl(USER_CACHE)
        by_url = {r["url"]: r for r in existing}
        for r in rows: by_url[r["url"]] = r
        merged = list(by_url.values())
        write_jsonl(USER_CACHE, merged)
        n = rebuild_user_index(merged)
        dt = time.time()-t0
        return f"✅ Загружено: {ok} файл(ов). В пользовательском индексе документов: {n}. ⏱ {dt:.1f}s", gr.update(value=[])

    def on_clear_docs():
        clear_user_index()
        return "🧹 Пользовательский индекс очищен.", gr.update(value=[])

    def on_clear_chat():
        global chat_history_store, last_sources_store, _last_q
        chat_history_store = []
        last_sources_store = []
        _last_q = {"text": None, "answer": None}
        return [], gr.update(visible=False), gr.update(visible=False)

    def on_ask(user_msg, history, use_web, only_user, only_ai, k, fres_lam, max_age, min_score):
        global last_sources_store, chat_history_store, _last_q
        if not user_msg or not str(user_msg).strip():
            return "", history

        # дубликат вопроса — отдаём прошлый ответ
        if _last_q["text"] and _clean_ws(user_msg) == _clean_ws(_last_q["text"]):
            prev = _last_q["answer"] or ""
            history = (history or []) + [{"role":"user","content": user_msg},
                                         {"role":"assistant","content": prev}]
            chat_history_store = history
            return "", history

        history = (history or []) + [{"role":"user","content": user_msg}]
        try:
            answer, sources = run_rag(
                user_msg,
                only_user=bool(only_user),
                only_ai_pool=bool(only_ai),
                use_web_toggle=bool(use_web),
                top_k=int(k),
                freshness_lambda=float(fres_lam),
                max_age_months=int(max_age),
                min_final_score=float(min_score),
            )
            src_md = "\n".join([f"- [{s['title']}]({s['url']}) ({s['date']})" for s in sources])
            md = f"{answer}\n\n**Источники:**\n{src_md}" if src_md else answer
        except Exception as e:
            md = f"⚠️ Ошибка при поиске/генерации: {e}"

        _last_q = {"text": user_msg, "answer": md}
        history = history + [{"role":"assistant","content": md}]
        chat_history_store = history
        last_sources_store = sources
        return "", history

    def on_report():
        path = save_docx_report(chat_history_store)
        return gr.update(visible=True, value=path)

    def on_download_sources():
        zip_path = download_sources_zip(last_sources_store)
        return gr.update(visible=True, value=(zip_path if zip_path else None))

    def on_bootstrap():
        # Генератор для индикации процесса
        total_new = 0
        by_url = {p["url"]: p for p in read_jsonl(AI_CACHE)}
        start_cnt = len(by_url)
        yield "🚀 Старт сборки пула про ИИ…"
        for i, q in enumerate(RU_AI_BOOTSTRAP_QUERIES, start=1):
            yield f"🔎 [{i}/{len(RU_AI_BOOTSTRAP_QUERIES)}] Поиск: **{q}**…"
            pages = fetch_pages_duckduckgo_ru_ai(q, max_pages=3)
            for p in pages:
                if p["url"] not in by_url:
                    total_new += 1
                by_url[p["url"]] = p
            yield f"📥 Найдено сейчас: {len(pages)}; всего новых: {total_new}…"
        merged = list(by_url.values())
        write_jsonl(AI_CACHE, merged)
        rebuild_ai_index(merged)
        yield f"✅ Готово. В пуле про ИИ: **{len(merged)}** страниц (новых: {total_new}). Индекс обновлён."

    def on_add_ai_urls(url_text: str):
        urls = [u.strip() for u in (url_text or "").splitlines() if u.strip()]
        if not urls:
            return "⚠️ Введите хотя бы один URL."
        added, total = add_urls_to_ai_pool(urls)
        return f"✅ Добавлено/обновлено URL: {added}. Всего в пуле: {total}."

    apply_evt = apply_btn.click(on_apply_llm, inputs=[provider, llm_model, llm_api_key, llm_base_url, hf_token], outputs=[llm_status])
    add_evt = add_btn.click(on_add, inputs=[file_uploader], outputs=[status_md, chatbot])
    clear_docs_btn.click(on_clear_docs, outputs=[status_md, chatbot])

    ask_evt = send_btn.click(
        on_ask,
        inputs=[msg, chatbot, web_toggle, only_user_toggle, only_ai_toggle, topk, freshness_lambda, max_age_months, min_final_score],
        outputs=[msg, chatbot]
    )
    stop_btn.click(fn=None, cancels=[ask_evt])

    clear_chat_btn.click(on_clear_chat, outputs=[chatbot, report_file, sources_file])
    report_btn.click(on_report, outputs=[report_file])
    download_btn.click(on_download_sources, outputs=[sources_file])
    bootstrap_btn.click(on_bootstrap, outputs=[ai_pool_status])
    add_url_btn.click(on_add_ai_urls, inputs=[add_url_text], outputs=[ai_pool_status])

# Одноразовая сборка пула про ИИ при старте (откл. по умолчанию)
if RUN_AI_POOL_BOOTSTRAP and not os.path.exists(AI_INDEX_DIR):
    try:
        # простая тихая сборка без прогресса
        by_url = {p["url"]: p for p in read_jsonl(AI_CACHE)}
        for q in RU_AI_BOOTSTRAP_QUERIES:
            for p in fetch_pages_duckduckgo_ru_ai(q, max_pages=2):
                by_url[p["url"]] = p
        merged = list(by_url.values())
        write_jsonl(AI_CACHE, merged)
        rebuild_ai_index(merged)
        print(f"✅ Пул про ИИ: {len(merged)} страниц.")
    except Exception as e:
        print("⚠️ Ошибка автосборки пула ИИ:", e)

demo.queue().launch(debug=False, share=True)
# =====================================================================

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.4/74.4 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.5/310.5 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.9/837.9 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.2/45.2 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 61.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 64.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/

/tmp/ipython-input-1776975791.py:209: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  emb = HuggingFaceEmbeddings(model_name=EMB_NAME_PRIMARY)
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://46711116eacdc58c1e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
